In [ ]:
# 构造链接预测数据集
# 直接就是在原始输入上面切段连接, 邻接矩阵!

# 读取数据
gt_path = '/home/pxl/myProject/血管分割/molong-深度插值/molong-work/simple-unet-2d/data/DRIVE/training/1st_manual/1st_manual.png'

# 读取全部图像:
import os
import cv2
import numpy as np
import torch
from no_one提取图结构 import segment_and_visualize_vessels,crop_image,visualize_segments_andNo,visualize_segment_graph,fill_small_holes
import 差分约束 as DifferentialConnection
import matplotlib.pyplot as plt
from scipy.spatial import distance
from skimage.measure import label, regionprops
import numpy as np
from PIL import Image
import cv2
import numpy as np
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
import matplotlib.pyplot as plt
import random
import sys
import networkx as nx
from scipy.ndimage import distance_transform_edt
from scipy.spatial import distance
from numpy.polynomial.polynomial import Polynomial
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling


# 获取文件夹中所有图像路径
img_dir = '/home/pxl/myProject/血管分割/molong-深度插值/molong-work/simple-unet-2d/data/DRIVE/training/1st_manual'
img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith('.gif')]

print(len(img_paths))
# 读取所有图像
i = 0
images = []
graphs_segments = []
shape = None

# 用于存储每个图构造的 Data 对象
data_list = []
for img_path in img_paths:
    i += 1
    if(i > 1): 
        break
    # size = (80, 60)  # 裁剪区域的尺寸 (width, height)
    position = (100, 160)  # 裁剪的起始位置 (x, y)
    size = (200, 100)  # 裁剪区域的尺寸 (width, height)
    cropped_image = crop_image(img_path, position, size,False)
    pred = np.array(cropped_image) 
    pred = fill_small_holes(pred)
    # plt.figure(figsize=(12, 8))
    # plt.imshow(pred, cmap='gray')

    _, binary = cv2.threshold(pred, 127, 255, cv2.THRESH_BINARY)
    segment_graph,skeleton,segments = segment_and_visualize_vessels(binary.copy(),False)
    graphs_segments.append((segment_graph,segments))
    shape = pred.shape
    
    node_features = []
    # 遍历 segment_graph 的节点和对应的特征
    for _, features in segment_graph.nodes(data=True):
        # 提取并转换特征
        endpoints = torch.tensor(features['endpoints'], dtype=torch.float)  # 假设 endpoints 是一个 (2, 2) 的坐标对
        length = torch.tensor([features['length']], dtype=torch.float)  # 长度特征
        pixel_count = torch.tensor([features['pixel_count']], dtype=torch.float)  # 像素计数特征

        # 将各特征拼接成一个节点特征向量
        node_feature = torch.cat([endpoints.view(-1), length, pixel_count], dim=0)  # 展平成一维
        node_features.append(node_feature)

    # 转换为节点特征矩阵 data.x
    data_x = torch.stack(node_features)  # shape: (num_nodes, feature_dim)
    
    
    # 假设 segment_graph 是 NetworkX 的 Graph 对象
    edges = list(segment_graph.edges)  # 获取所有边，形式为 [(src, dst), ...]
    # 将边列表转换为 edge_index 张量
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()  # shape: (2, num_edges)
    # 打印 edge_index 检查结果
    # 获取节点数量
    num_nodes = segment_graph.number_of_nodes()
    # 创建 Data 对象并添加到 data_list 中
    data = Data(x=data_x, edge_index=edge_index, num_nodes=num_nodes)
    data_list.append(data)
    
    
    masked_data = data.clone()
    # 计算要移除的边数量
    num_edges = data.edge_index.size(1)
    num_mask = int(num_edges * 0.1)
    # 随机选择要保留和移除的边
    perm = torch.randperm(num_edges)
    mask_indices = perm[:num_mask]
    remain_indices = perm[num_mask:]
    
    # 保存原始边用于生成标签
    original_edges = data.edge_index.clone()
    
    # 更新图的边（移除被掩码的边）
    masked_data.edge_index = data.edge_index[:, remain_indices]
    
    # 生成负样本边（使用negative_sampling）
    neg_edge_index = negative_sampling(
        edge_index=original_edges,
        num_nodes=data.num_nodes,
        num_neg_samples=num_mask,
        method='sparse'
    )
    
    # 构造edge_label_index，包含:
    # 1. 被移除的真实边（正样本）
    # 2. 负采样生成的边（负样本）
    edge_label_index = torch.cat([
        original_edges[:, mask_indices],  # 被掩码的真实边
        neg_edge_index  # 负采样边
    ], dim=1)
    
    # 构造对应的标签
    edge_label = torch.cat([
        torch.ones(num_mask),    # 真实边的标签为1
        torch.zeros(num_mask)    # 负采样边的标签为0
    ])
    
    print(data)
    print(edge_index)
    print(edge_label_index)
    print(edge_label)



In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_dense_adj, remove_self_loops, dense_to_sparse

from torch_geometric.datasets import Planetoid

# 设置国内镜像源
dataset = Planetoid(
    root='/tmp/Cora',
    name='Cora',
    url='https://mirrors.tuna.tsinghua.edu.cn/torch-geometric/datasets/Cora.zip'
)

graph = dataset[0]

# GAE 编码器
class GAEEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, latent_channels):
        super(GAEEncoder, self).__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn_latent = GCNConv(hidden_channels, latent_channels)

    def forward(self, x, edge_index):
        h = F.relu(self.gcn1(x, edge_index))
        z = self.gcn_latent(h, edge_index)
        return z

# GAE 模型
class GAE(nn.Module):
    def __init__(self, encoder):
        super(GAE, self).__init__()
        self.encoder = encoder

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        return z

    def decode(self, z, edge_index):
        # 解码器：计算边的存在概率
        logits = (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)
        return logits

    def loss(self, pos_logits):
        # 只计算正样本的重构损失
        loss = -torch.log(torch.sigmoid(pos_logits) + 1e-8).mean()
        return loss

# 构造训练集和测试集
def prepare_data(graph, test_ratio=0.2):
    edge_index = graph.edge_index
    num_edges = edge_index.size(1)

    # 随机拆分边
    perm = torch.randperm(num_edges)
    num_test = int(test_ratio * num_edges)
    test_edges = edge_index[:, perm[:num_test]]  # 测试集边
    train_edges = edge_index[:, perm[num_test:]]  # 训练集边

    # 构造部分观测图（训练图）
    train_graph = graph.clone()
    train_graph.edge_index = train_edges

    # 移除自环
    train_graph.edge_index, _ = remove_self_loops(train_graph.edge_index)

    return train_graph, train_edges, test_edges

# 模型训练和测试
def train_and_evaluate(graph, epochs=100, lr=0.01, hidden_channels=64, latent_channels=32):
    # 准备数据
    train_graph, train_edges, test_edges = prepare_data(graph)

    # 模型定义
    encoder = GAEEncoder(graph.num_features, hidden_channels, latent_channels)
    model = GAE(encoder)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 训练
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        z = model(train_graph.x, train_graph.edge_index)

        # 正样本解码
        pos_logits = model.decode(z, train_edges)

        # 损失计算
        loss = model.loss(pos_logits)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")

    # 测试
    with torch.no_grad():
        model.eval()
        z = model(train_graph.x, train_graph.edge_index)

        # 测试集正样本预测
        test_pos_logits = model.decode(z, test_edges)

        # ROC-AUC 和 Average Precision
        from sklearn.metrics import roc_auc_score, average_precision_score
        y_true = torch.ones(test_pos_logits.size(0))  # 全部为正样本
        y_pred = test_pos_logits.cpu()

        roc_auc = roc_auc_score(y_true, y_pred)
        ap_score = average_precision_score(y_true, y_pred)

        print(f"\nTest ROC-AUC: {roc_auc:.4f}, AP: {ap_score:.4f}")

# 主流程
train_and_evaluate(graph)


TypeError: __init__() got an unexpected keyword argument 'url'